# BOW and TFIDF

## Exploring Changing Sentiment in Climate Articles: 2013 to 2023

#### This notebook explores BOW and TFIDF concepts within the corpus.

#### Written by Rafael Alvarado(1) and Caroline Kranefuss(1).

(1) University of Virginia, 2026

### Imports

In [20]:
# General imports
import pandas as pd 
import numpy as np 
import os
import sys

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Project-specific imports
import glob
from lxml import etree
from glob import glob
import re
import nltk
nltk_resources = [
    'tokenizers/punkt', 
    'averaged_perceptron_tagger_eng',
    'corpora/stopwords', 
    'help/tagsets'
]

for resource in nltk_resources:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(resource)
        
        
# ----File Stitching----
# If not in repo home folder, cd back 
if os.path.basename(os.getcwd()) != "evolving_sentiment_climate":
    os.chdir('..')
# If a file is in /sources/, access it by telling the system to look at that path as well as current path
sys.path.append(os.path.join(os.getcwd(), 'sources'))
source_dir = "sources"
source_files_paths = glob(f"{source_dir}/*.xml")
# Same for csvs
sys.path.append(os.path.join(os.getcwd(), 'csvs'))
csvs_dir = "csvs"
csvs_files_paths = glob(f"{csvs_dir}/*.csv")

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\student\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [13]:
sns.set_theme(style='darkgrid')

### Defining OHCO

In [27]:
# Define OHCO
OHCO = ['year', 'mth_day', 'doc_id', 'sent_num', 'token_num']
bags = dict(
    SENTS = OHCO[:4],
    DOCS = OHCO[:3],
    MTH_DAY = OHCO[:2],
    YEAR = OHCO[:1]
)

bag = 'YEAR' # Goal is to explore by year

### Importing Tables

In [52]:
TOKENS = pd.read_csv("csvs/TOKENS/TOKENS_final.csv").set_index(OHCO).dropna()
LIB = pd.read_csv("csvs/LIB/LIB.csv").set_index("year")
VOCAB = pd.read_csv("csvs/VOCAB/VOCAB_final.csv").set_index("term_str").dropna()

### Creating BOW, DTCM from TOKENS

In [31]:
# Define BOW by grouping TOKENS by the proper bag and adding counts
BOW = TOKENS.groupby(bags[bag]+['term_str']).term_str.count().to_frame('n') 
BOW.head()

n
year term_str    
2013 0         21
     000        3
     0000042    1
     00001      9
     0000152    1

In [32]:
# Unstack the BOW to create a Document-Term Coutn Matrix
DTCM = BOW.n.unstack(fill_value=0)
DTCM.head(10)

term_str,0,00,000,0000007,000003,0000042,00001,0000152,00002,00003,...,ω,ωarag,ωcal,ϕ,ϕph2oph2o,ϕst,ϵ,ϵerror,ϵn,ϵy
year,,,,,,,,,,,,,,,,,,,,,
2013,21,0,3,0,0,1,9,1,1,0,...,2,0,0,0,0,0,1,1,1,0
2018,80,3,1,0,1,0,8,0,0,0,...,5,35,1,0,0,5,1,0,0,1
2023,69,1,3,1,0,0,2,0,0,4,...,0,0,0,1,1,0,0,0,0,0


### Computing TF, DF, IDF, TFIDF

In [37]:
#------------ Compute TF --------------

# Settings: options are sum, max, log, double_norm, raw, binary, I will use double_norm which requires k
tf_method = 'double_norm'        
tf_norm_k = .5          
# Options are standard, max, smooth 
idf_method = 'standard'  
gradient_cmap = 'YlGnBu' 

BOW.groupby(['year']).apply(lambda x: x.n / x.n.sum())

TF = ((DTCM.T + .5) / (DTCM.T.max() + .5)) + .5 # Is this correct?

# Take transpose
TF = TF.T

# View
TF.head()

term_str,0,00,000,0000007,000003,0000042,00001,0000152,00002,00003,...,ω,ωarag,ωcal,ϕ,ϕph2oph2o,ϕst,ϵ,ϵerror,ϵn,ϵy
year,,,,,,,,,,,,,,,,,,,,,
2013,0.504332,0.500101,0.500705,0.500101,0.500101,0.500302,0.501914,0.500302,0.500302,0.500101,...,0.500504,0.500101,0.500101,0.500101,0.500101,0.500101,0.500302,0.500302,0.500302,0.500101
2018,0.506884,0.500299,0.500128,0.500043,0.500128,0.500043,0.500727,0.500043,0.500043,0.500043,...,0.500470,0.503036,0.500128,0.500043,0.500043,0.500470,0.500128,0.500043,0.500043,0.500128
2023,0.504775,0.500103,0.500240,0.500103,0.500034,0.500034,0.500172,0.500034,0.500034,0.500309,...,0.500034,0.500034,0.500034,0.500103,0.500103,0.500034,0.500034,0.500034,0.500034,0.500034


In [39]:
# ------------- Compute DF ------------

DF = DTCM.astype('bool').sum() 
DF

term_str
0          3
00         2
000        3
0000007    1
000003     1
          ..
ϕst        1
ϵ          2
ϵerror     1
ϵn         1
ϵy         1
Length: 27367, dtype: int64

In [42]:
# ---------------- Compute IDF -------------

IDF = np.log2(DTCM.shape[0] / DF)
IDF

term_str
0          0.000000
00         0.584963
000        0.000000
0000007    1.584963
000003     1.584963
             ...   
ϕst        1.584963
ϵ          0.584963
ϵerror     1.584963
ϵn         1.584963
ϵy         1.584963
Length: 27367, dtype: float64

In [43]:
# -------------- Compute TFIDF ----------

TFIDF = TF * IDF
TFIDF.head()

term_str,0,00,000,0000007,000003,0000042,00001,0000152,00002,00003,...,ω,ωarag,ωcal,ϕ,ϕph2oph2o,ϕst,ϵ,ϵerror,ϵn,ϵy
year,,,,,,,,,,,,,,,,,,,,,
2013,0.0,0.292540,0.0,0.792641,0.792641,0.792960,0.0,0.792960,0.792960,0.792641,...,0.292776,0.792641,0.792641,0.792641,0.792641,0.792641,0.292658,0.792960,0.792960,0.792641
2018,0.0,0.292656,0.0,0.792549,0.792685,0.792549,0.0,0.792549,0.792549,0.792549,...,0.292756,0.797293,0.792685,0.792549,0.792549,0.793227,0.292556,0.792549,0.792549,0.792685
2023,0.0,0.292542,0.0,0.792645,0.792536,0.792536,0.0,0.792536,0.792536,0.792971,...,0.292501,0.792536,0.792536,0.792645,0.792645,0.792536,0.292501,0.792536,0.792536,0.792536


In [71]:
TFIDF.mean()

term_str
0          0.000000
00         0.292579
000        0.000000
0000007    0.792612
000003     0.792620
             ...   
ϕst        0.792801
ϵ          0.292572
ϵerror     0.792682
ϵn         0.792682
ϵy         0.792620
Length: 27367, dtype: float64

In [76]:
# Organization
VOCAB['df'] = DF
VOCAB['idf'] = IDF
VOCAB['tfidf_mean'] = TFIDF.mean()

In [77]:
VOCAB

,year,n,p,i,p_yr,i_yr,n_pos_group,cat_pos_group,n_pos,cat_pos,max_pos,max_pos_group,df,idf,tfidf_mean
term_str,,,,,,,,,,,,,,,
the,2013,4963,0.009662,6.693500,0.059097,4.080779,2,"{'VB', 'DT'}",2,"{'VBP', 'DT'}",DT,DT,3.0,0.000000,0.000000
of,2013,3106,0.006047,7.369654,0.036985,4.756933,3,"{'NN', 'IN', 'CC'}",3,"{'NN', 'IN', 'CC'}",IN,IN,3.0,0.000000,0.000000
and,2013,2631,0.005122,7.609101,0.031329,4.996380,2,"{'NN', 'CC'}",2,"{'NNP', 'CC'}",CC,CC,3.0,0.000000,0.000000
in,2013,2310,0.004497,7.796819,0.027506,5.184098,2,"{'NN', 'IN'}",2,"{'NNP', 'IN'}",IN,IN,3.0,0.000000,0.000000
to,2013,1710,0.003329,8.230716,0.020362,5.617995,2,"{'TO', 'NN'}",2,"{'NNP', 'TO'}",TO,TO,3.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
scpi,2023,1,0.000002,18.970496,0.000004,17.864035,1,{'NN'},1,{'NNP'},NNP,NN,1.0,1.584963,0.792612
fba,2023,1,0.000002,18.970496,0.000004,17.864035,1,{'NN'},1,{'NNP'},NNP,NN,1.0,1.584963,0.792612
sip11,2023,1,0.000002,18.970496,0.000004,17.864035,1,{'NN'},1,{'NNP'},NNP,NN,1.0,1.584963,0.792612


In [83]:
def top_years_for_term(term_str):
    return TFIDF[term_str]\
        .to_frame('tfidf_mean')\
        .join(LIB)\
        .drop_duplicates(keep='first')\
        .sort_values('tfidf_mean', ascending=False)\
        ['tfidf_mean']\
        #.style.bar()

top_years_for_term('data')

year
2013    0.0
2013    0.0
2013    0.0
2013    0.0
2013    0.0
       ... 
2023    0.0
2023    0.0
2023    0.0
2023    0.0
2023    0.0
Name: tfidf_mean, Length: 92, dtype: float64